# ML Systems 101: From Jupyter to Production
## Live Demo — Instructor-Run
**ML System Design and LLMOps | Week 1, Session 1**

This notebook is the instructor's live demo. We'll walk through three things:
1. The "notebook version" of an ML pipeline — and what it's missing
2. How training-serving skew silently breaks a deployed model
3. A sketch of what a real six-component ML system looks like in code

## Part 1: The Notebook Version
*"We trained a model. It got 94% accuracy. Let's ship it."*

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test_scaled))
print(f"Test accuracy: {accuracy:.3f}")
print("\nDone. Ship it? 🚀")

Test accuracy: 0.974

Done. Ship it? 🚀


## Part 1b: What We Just Did (and Didn't Do)

✅ Trained a model  
✅ Evaluated on a test set  
❌ Saved the scaler  
❌ Saved the model  
❌ Versioned either of them  
❌ Written a serving function  
❌ Defined what "good" looks like at month 3  

**The question isn't "does it work?" — it's "does it keep working?"**

## Part 2: Training-Serving Skew
*The #1 silent killer in production ML*

In [2]:
import pickle
import numpy as np

# Save ONLY the model — not the scaler (common mistake)
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# --- Later, at "serving time" ---
# A new engineer picks up the model. They don't know about the scaler.

with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# They predict on RAW (unscaled) test data
y_pred_broken = loaded_model.predict(X_test)  # no scaler applied!
broken_accuracy = accuracy_score(y_test, y_pred_broken)

print(f"Original accuracy (scaled):   {accuracy:.3f}")
print(f"Serving accuracy (unscaled):  {broken_accuracy:.3f}")
print(f"Silent accuracy drop:         {accuracy - broken_accuracy:.3f}")
print()
print("No error was raised. No exception. No alert.")
print("The model returned predictions. They were just wrong.")

Original accuracy (scaled):   0.974
Serving accuracy (unscaled):  0.377
Silent accuracy drop:         0.596

No error was raised. No exception. No alert.
The model returned predictions. They were just wrong.


## Part 2b: What Actually Happened

The model was trained on **standardized** features (mean=0, std=1).  
At serving time, it received **raw** features (mean≈14, std≈7 for some columns).  

The model had no way to know this was wrong.  
It confidently returned incorrect predictions.  

**This is training-serving skew. It is silent. It is common. It is preventable.**

The fix: save the scaler alongside the model. Better: use a Pipeline object.

In [3]:
from sklearn.pipeline import Pipeline

# The right way: bundle preprocessing with the model
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)

# Save the entire pipeline
with open("pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

# Load and serve — scaler is automatically applied
with open("pipeline.pkl", "rb") as f:
    loaded_pipeline = pickle.load(f)

pipeline_accuracy = accuracy_score(y_test, loaded_pipeline.predict(X_test))
print(f"Pipeline accuracy (raw input → correct prediction): {pipeline_accuracy:.3f}")
print("Training-serving skew: eliminated.")

Pipeline accuracy (raw input → correct prediction): 0.974
Training-serving skew: eliminated.


## Part 3: A Six-Component ML System (Sketch)
*What production actually looks like — in Python*

In [4]:
import time
from collections import deque
from datetime import datetime

# ── 1. Data Pipeline ─────────────────────────────────────────────────────────
def data_pipeline(user_id: str) -> dict:
    """Fetch and validate raw features for a user. In prod: reads from Kafka/S3."""
    raw = {
        "user_id": user_id,
        "feature_1": 14.2,
        "feature_2": -0.3,
        "timestamp": datetime.utcnow().isoformat(),
    }
    assert "user_id" in raw, "Schema validation failed"
    return raw

# ── 2. Feature Store ─────────────────────────────────────────────────────────
feature_store = {}  # In prod: Redis (online) + S3/Parquet (offline)

def update_feature_store(user_id: str, features: dict):
    feature_store[user_id] = features

def get_features(user_id: str) -> dict:
    return feature_store.get(user_id, {})

# ── 3. Training Pipeline ─────────────────────────────────────────────────────
model_registry = {}  # In prod: MLflow Model Registry

def train_model(version: str = "v1"):
    """Train and register a model. In prod: scheduled Airflow/Prefect job."""
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])
    pipe.fit(X_train, y_train)
    model_registry[version] = pipe
    print(f"[Registry] Model {version} registered at {datetime.utcnow().isoformat()}")
    return version

# ── 4. Model Registry ─────────────────────────────────────────────────────────
production_model_version = None

def promote_to_production(version: str):
    global production_model_version
    production_model_version = version
    print(f"[Registry] {version} → PRODUCTION")

# ── 5. Serving Layer ─────────────────────────────────────────────────────────
prediction_log = deque(maxlen=100)  # rolling window for monitoring

def serve(raw_features: dict) -> dict:
    """Real-time prediction endpoint. In prod: REST/gRPC behind a load balancer."""
    assert production_model_version, "No model in production!"
    model = model_registry[production_model_version]
    
    feature_vector = [[raw_features["feature_1"], raw_features["feature_2"]] + [0]*28]
    # (simplified — real serving would reconstruct the full feature vector)
    
    prediction = int(model.predict([[*data.data[0]]])[0])  # use a real sample
    confidence = float(model.predict_proba([[*data.data[0]]])[0].max())
    
    result = {"prediction": prediction, "confidence": confidence, "model": production_model_version}
    prediction_log.append(result)
    return result

# ── 6. Monitoring ─────────────────────────────────────────────────────────────
def monitor():
    """Check system health. In prod: runs on a schedule, writes to dashboards."""
    if not prediction_log:
        print("[Monitor] No predictions yet.")
        return
    avg_confidence = sum(p["confidence"] for p in prediction_log) / len(prediction_log)
    print(f"[Monitor] Predictions logged: {len(prediction_log)}")
    print(f"[Monitor] Avg confidence:     {avg_confidence:.3f}")
    print(f"[Monitor] Active model:       {production_model_version}")
    if avg_confidence < 0.7:
        print("[Monitor] ⚠️  LOW CONFIDENCE — investigate feature drift")

In [5]:
# Run the full system
print("=" * 50)
print("RUNNING THE FULL ML SYSTEM")
print("=" * 50)

# Step 1: Ingest data
raw = data_pipeline("user_42")
update_feature_store("user_42", raw)
print(f"[Pipeline] Features ingested for user_42: {list(raw.keys())}")

# Step 2: Train + register
version = train_model("v1")

# Step 3: Promote to production
promote_to_production("v1")

# Step 4: Serve 5 predictions
for i in range(5):
    features = get_features("user_42")
    result = serve(features)
    print(f"[Serving]  Prediction {i+1}: class={result['prediction']}, confidence={result['confidence']:.3f}")

# Step 5: Monitor
print()
monitor()

RUNNING THE FULL ML SYSTEM
[Pipeline] Features ingested for user_42: ['user_id', 'feature_1', 'feature_2', 'timestamp']
[Registry] Model v1 registered at 2026-08-19T21:00:39.047010
[Registry] v1 → PRODUCTION
[Serving]  Prediction 1: class=0, confidence=1.000
[Serving]  Prediction 2: class=0, confidence=1.000
[Serving]  Prediction 3: class=0, confidence=1.000
[Serving]  Prediction 4: class=0, confidence=1.000
[Serving]  Prediction 5: class=0, confidence=1.000

[Monitor] Predictions logged: 5
[Monitor] Avg confidence:     1.000
[Monitor] Active model:       v1


## Part 4: Discussion

**What would break first if we deployed this to 1 million users?**

Take 2 minutes — write your answer before we discuss.

---

**What's missing from our `monitor()` function?**

Hint: we track confidence. What else matters?

---

*These questions don't have one right answer. They're the questions you'll answer in every system design session this term.*